<a href="https://colab.research.google.com/github/JuanFerMC/TeoriaDelLenguaje-LABS/blob/main/Lab_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio #6
---
##Teoria de lenguaje y laboratorio - 2026/1
### Hecho por:
* Juan Fernando Mona Cano | juanf.mona@udea.edu.co
* Ulises Orozco Villegas | ulises.orozco@udea.edu.co

### Docente:
* Ruben Dario Angel Correa


In [ ]:
# IMPORTS
import graphviz
from IPython.display import display, clear_output
import time
import ipywidgets as widgets

# CLASES DEL ÁRBOL
class ParseTreeNode:
    def __init__(self, node_type, children=None, value=None):
        self.node_type = node_type
        self.children = children or []
        self.value = value
        self.attr_valor = None
        self.attr_tipo = None
        self.id = str(id(self))

    def add_child(self, child_node):
        self.children.append(child_node)

class ExprNode(ParseTreeNode):
    def __init__(self, children=None):
        super().__init__('expr', children)

class NumNode(ParseTreeNode):
    def __init__(self, children=None):
        super().__init__('num', children)

class TerminalNode(ParseTreeNode):
    def __init__(self, value):
        super().__init__('terminal', value=value)

#AST (parser simple)
class ASTNode:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def build_ast(expression):
    tokens = expression.replace('+', ' + ').replace('-', ' - ') \
        .replace('*', ' * ').replace('/', ' / ').split()

    def parse_expression(tokens):
        left = parse_term(tokens)
        while tokens and tokens[0] in ('+', '-'):
            op = tokens.pop(0)
            right = parse_term(tokens)
            left = ASTNode(op, left, right)
        return left

    def parse_term(tokens):
        left = parse_factor(tokens)
        while tokens and tokens[0] in ('*', '/'):
            op = tokens.pop(0)
            right = parse_factor(tokens)
            left = ASTNode(op, left, right)
        return left

    def parse_factor(tokens):
        return ASTNode(tokens.pop(0))

    return parse_expression(tokens)

#AST → Árbol de derivación
def convert_ast_to_parse_tree(ast_node):
    if not ast_node:
        return None

    if ast_node.left is None and ast_node.right is None:
        num_node = NumNode()
        num_node.add_child(TerminalNode(ast_node.value))

        expr_node = ExprNode()
        expr_node.add_child(num_node)
        return expr_node
    else:
        expr_node = ExprNode()
        expr_node.add_child(convert_ast_to_parse_tree(ast_node.left))
        expr_node.add_child(TerminalNode(ast_node.value))
        expr_node.add_child(convert_ast_to_parse_tree(ast_node.right))
        return expr_node

#DIBUJO DEL ÁRBOL
def draw_parse_tree(root, highlight_ids=None, evaluated=None, visible_ids=None):
    dot = graphviz.Digraph(format='svg')
    dot.attr(rankdir='TB', bgcolor="#0f172a")

    dot.attr('node',
             shape='circle',
             style='filled',
             fontname="Helvetica",
             fontsize="10",
             color="#334155")

    highlight_ids = highlight_ids or []
    evaluated = evaluated or {}
    visible_ids = visible_ids or set()

    def add_nodes(n):
        if not n or (visible_ids and n.id not in visible_ids):
            return

        #Colores
        base = "#1e293b"
        expr_c = "#60a5fa"
        num_c = "#38bdf8"
        highlight_c = "#f59e0b"
        eval_c = "#22c55e"
        fill = base
        label = ""
        font_color = "white"

        if n.node_type == 'expr':
            fill = expr_c
            label = "expr"

        elif n.node_type == 'num':
            fill = num_c
            label = "num"

        elif n.node_type == 'terminal':
            fill = "#e2e8f0"
            label = str(n.value)
            font_color = "black"

        # Resaltado
        if n.id in highlight_ids:
            fill = highlight_c

        # Evaluación (tipo + valor)
        if n.id in evaluated:
            fill = eval_c
            if n.node_type != 'terminal':
                val = evaluated[n.id].get("valor", "")
                tipo = evaluated[n.id].get("tipo", "")
                label = f"{label}\n{tipo}\n{val}"

        dot.node(n.id, label=label, fillcolor=fill, fontcolor=font_color)

        for child in n.children:
            if not visible_ids or child.id in visible_ids:
                dot.edge(n.id, child.id, color="#94a3b8")
            add_nodes(child)

    add_nodes(root)
    return dot

# ANIMACIÓN
def animate_parse_tree(expression):
    try:
        ast_root = build_ast(expression)
        pt_root = convert_ast_to_parse_tree(ast_root)
    except Exception as e:
        print("Error:", e)
        return

    steps = []
    visible = set()

    #Construcción progresiva
    def collect_structure(n):
        if n:
            visible.add(n.id)
            steps.append({
                'highlight': [n.id],
                'eval': {},
                'visible': visible.copy()
            })
            for child in n.children:
                collect_structure(child)

    #Evaluación
    eval_map = {}
    def evaluate(n):
        if n.node_type == 'terminal':
            return None

        if n.node_type == 'num':
            val = float(n.children[0].value)
            tipo = "int" if val.is_integer() else "float"
            eval_map[n.id] = {"valor": val, "tipo": tipo}
            steps.append({
                'highlight': [n.id],
                'eval': eval_map.copy(),
                'visible': visible.copy()
            })
            return val

        if n.node_type == 'expr':

            if len(n.children) == 1:
                val = evaluate(n.children[0])
                tipo = "int" if float(val).is_integer() else "float"
                eval_map[n.id] = {"valor": val, "tipo": tipo}
                steps.append({
                    'highlight': [n.id],
                    'eval': eval_map.copy(),
                    'visible': visible.copy()
                })
                return val

            left = evaluate(n.children[0])
            op = n.children[1].value
            right = evaluate(n.children[2])
            if op == '+': res = left + right
            elif op == '-': res = left - right
            elif op == '*': res = left * right
            elif op == '/': res = left / right if right != 0 else float('inf')

            tipo = "int" if float(res).is_integer() else "float"
            eval_map[n.id] = {"valor": round(res, 2), "tipo": tipo}
            steps.append({
                'highlight': [n.id],
                'eval': eval_map.copy(),
                'visible': visible.copy()
            })

            return res

    collect_structure(pt_root)
    evaluate(pt_root)

    #Mostrar animación
    out = widgets.Output()
    display(out)
    for i, step in enumerate(steps):
        with out:
            clear_output(wait=True)
            print(f"Paso {i+1}/{len(steps)} → {expression}")
            display(draw_parse_tree(
                pt_root,
                step['highlight'],
                step['eval'],
                step['visible']
            ))
        time.sleep(0.8 if i < len(steps)//2 else 0.8)

#EJECUCIÓN
print("Ejemplo: 9 + 7 * 5")
exp = input("Ingresa expresión: ")
animate_parse_tree(exp)

Ejemplo: 9 + 7 * 5
Ingresa expresión: 9 + 7 * 5


Output()